# Fusion Perception — Pipeline → Fine-tune Gemma 4 2B

End-to-end notebook:
1. **Sections 1–2** — set up environment, run the perception pipeline on KITTI-360, collect `reasoning.json` outputs
2. **Section 3** — extract and inspect the training dataset
3. **Section 4** — fine-tune Gemma 4 2B with QLoRA (HuggingFace PEFT, no Unsloth)
4. **Section 5** — evaluate the fine-tuned adapter

**Before running:** Runtime → Change runtime type → **T4 GPU** (or L4 if available)

## Section 1 — Environment

In [1]:
# Cell 1 — Mount Google Drive and navigate to project
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/Fusion_sensor'   # ← edit if needed
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

Mounted at /content/drive
Working directory: /content/drive/MyDrive/Fusion_sensor


In [ ]:
# Cell 2 — Install all dependencies

import os
os.chdir('/content')   # prevent Drive FUSE OSError 107 during large pip writes

# 0 — pin setuptools
!pip install "setuptools<82" -q

# 1 — DO NOT reinstall torch. Use Colab's preinstalled build.
import torch
print(f"Runtime torch: {torch.__version__}")

# 2 — transformers upgrade (required for Gemma 4)
!pip install --upgrade transformers -q

# 3 — all project + fine-tuning deps (pure pip, no compilation)
!pip install omegaconf>=2.3.0 opencv-python>=4.9.0 Pillow>=10.0.0 \
    dataclasses-json>=0.6.0 rich>=13.0.0 \
    huggingface_hub>=0.26.0 transformers>=4.47.0 \
    bitsandbytes>=0.44.0 accelerate>=1.2.0 \
    peft>=0.14.0 trl>=0.12.0 datasets>=3.0.0 \
    matplotlib>=3.8.0 filterpy>=1.4.5 \
    portalocker einops timm \
    ultralytics>=8.3.0 open3d>=0.18.0 kitti360scripts -q

# 4 — CoWTracker (optional — falls back to KF-only if this fails)
!pip install git+https://github.com/facebookresearch/co-tracker -q \
    || echo "WARNING: CoWTracker install failed — tracker will run in KF-only mode."

# 5 — verify ultralytics (used by FrustumLidarDetector and RoadSegmentor)
import importlib; importlib.invalidate_caches()
try:
    from ultralytics import YOLO
    print(f"ultralytics ready.")
except ImportError as e:
    raise RuntimeError(f"ultralytics not importable: {e}")

# 6 — repair Colab system packages that the installs above may downgrade
!pip install -U requests==2.32.4 urllib3 tqdm rich -q

# 7 — local project package
_project_dir = '/content/drive/MyDrive/Fusion_sensor'
!pip install -e {_project_dir} -q

try:
    os.chdir(_project_dir)
    print(f'Working directory: {os.getcwd()}')
except OSError:
    print('Drive not mounted — run cell-drive first, then re-run this cell')

print('Installation complete.')

In [ ]:
# Cell 3 — Verify GPU
import torch

assert torch.cuda.is_available(), 'No GPU — switch runtime to T4'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {vram_gb:.1f} GB')
assert vram_gb >= 10.0, f'Need >= 10 GB VRAM, got {vram_gb:.1f} GB'
print('GPU check passed')

## Section 2 — Run Perception Pipeline

Runs the full stack (CenterPoint LiDAR detector → CoWTracker → BEV → Gemma 4 visual reasoning)
on KITTI-360. Each Gemma call saves `prompt_used` + `summary` to `reasoning.json` —
this becomes the fine-tuning dataset.

Use `pipeline_inference.ipynb` for the full pipeline run, then come back here for fine-tuning.

In [ ]:
# Cell 4 — Pipeline configuration
import os, datetime
import numpy as np
from pathlib import Path

# ── Data source ──────────────────────────────────────────────────────────
# Set DATA_FORMAT to 'raw' for KITTI raw, '360' for KITTI-360.
DATA_FORMAT   = '360'                              # 'raw' | '360'

# KITTI raw (only used when DATA_FORMAT='raw')
DATA_PATH_ROOT = '2011_10_03'
DRIVE_SEQ      = '2011_10_03_drive_0047_sync'

# KITTI-360 (only used when DATA_FORMAT='360')
KITTI360_ROOT = '/content/drive/MyDrive/Fusion_sensor/KITTI-360'
KITTI360_SEQ  = '2013_05_28_drive_0000_sync'

MAX_FRAMES    = 500   # set to None for full sequence   # None = full sequence; int to cap
ENCODE_VIDEO  = False  # True to encode MP4 (slow, ~10min); False to skip
REASONING_INTERVAL = 15

VIDEO_PATH = (
    f'/content/kitti360_{KITTI360_SEQ}.mp4' if DATA_FORMAT == '360'
    else '/content/kitti_0047.mp4'
)

run_id  = datetime.datetime.now().strftime('%Y-%m-%d_%H%M%S') + '_' + (
    KITTI360_SEQ if DATA_FORMAT == '360' else DRIVE_SEQ
)
out_dir = Path('outputs') / run_id
out_dir.mkdir(parents=True, exist_ok=True)
print(f'Run ID     : {run_id}')
print(f'Output     : {out_dir}')
print(f'Data format: {DATA_FORMAT}')

SHOW_DEBUG        = True
DEBUG_EVERY       = 50
ENABLE_GT_EVAL    = True
SAVE_DEBUG_FRAMES = True


In [ ]:
# Cell 5 — Prepare checkpoint directory
#
# CenterPoint KITTI weights (~83 MB) are downloaded automatically on first
# run by CenterPointDetector.load() → saved to ckpt/centerpoint_kitti_3class.pth
# and reused on subsequent runs.
#
# No manual download needed.

import os
os.makedirs('ckpt', exist_ok=True)
print('ckpt/ directory ready.')
print('CenterPoint checkpoint will be downloaded automatically on first detector.load().')


In [ ]:
# Cell 6 — Download KITTI raw data (skipped if already on Drive)
import os
seq_dir = os.path.join(DATA_PATH_ROOT, DRIVE_SEQ)
if not os.path.exists(seq_dir):
    !wget -q https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_10_03_drive_0047/2011_10_03_drive_0047_sync.zip
    !wget -q https://s3.eu-central-1.amazonaws.com/avg-kitti/raw_data/2011_10_03_calib.zip
    !jar xf 2011_10_03_drive_0047_sync.zip
    !jar xf 2011_10_03_calib.zip
    print('Download complete.')
else:
    print('Data already present — skipping download.')

In [ ]:
# Cell 6b — Download KITTI-360 data directly from official S3 storage
#
# Saves to Google Drive for persistence across sessions.
# Set DOWNLOAD_LIDAR = True to also fetch LiDAR (~3-5 GB per sequence).

import os, zipfile, glob, subprocess, re

KITTI360_ROOT  = '/content/drive/MyDrive/Fusion_sensor/KITTI-360'
KITTI360_SEQ   = '2013_05_28_drive_0000_sync'
DOWNLOAD_LIDAR = True

S3 = 'https://s3.eu-central-1.amazonaws.com/avg-projects/KITTI-360'

os.makedirs(KITTI360_ROOT, exist_ok=True)

def _wget(url, dest_dir):
    name = url.split('/')[-1].split('?')[0]
    out  = os.path.join(dest_dir, name)
    if os.path.exists(out):
        print(f'  already exists: {name}')
        return out
    print(f'  downloading {name} ...')
    r = subprocess.run(['wget', '-c', '--show-progress', '-P', dest_dir, url])
    if r.returncode != 0:
        raise RuntimeError(f'wget failed (exit {r.returncode}) for {url}')
    return out

def _unzip(path, dest, show_contents=False):
    print(f'  extracting {os.path.basename(path)} -> {os.path.basename(dest)}/')
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(path) as zf:
        if show_contents:
            print(f'  first entries: {zf.namelist()[:3]}')
        zf.extractall(dest)
    os.remove(path)

def _probe_url(url):
    """Return True if the URL exists (HTTP 200/206)."""
    r = subprocess.run(['wget', '--spider', '-q', url], capture_output=True)
    return r.returncode == 0

# ── 1. Calibration ────────────────────────────────────────────────────────────
if os.path.isfile(os.path.join(KITTI360_ROOT, 'calibration', 'perspective.txt')):
    print('calibration: already present')
else:
    z = _wget(f'{S3}/384509ed5413ccc81328cf8c55cc6af078b8c444/calibration.zip', KITTI360_ROOT)
    _unzip(z, KITTI360_ROOT)
    print('calibration: done')

# ── 2. Vehicle poses (zip is flat: {seq}/poses.txt — extract into data_poses/) ─
poses_file = os.path.join(KITTI360_ROOT, 'data_poses', KITTI360_SEQ, 'poses.txt')
if os.path.isfile(poses_file):
    print('data_poses: already present')
else:
    z = _wget(f'{S3}/89a6bae3c8a6f789e12de4807fc1e8fdcf182cf4/data_poses.zip', KITTI360_ROOT)
    _unzip(z, os.path.join(KITTI360_ROOT, 'data_poses'), show_contents=True)
    print('data_poses: done')

# ── 3. 3D bounding boxes ──────────────────────────────────────────────────────
bbox_file = os.path.join(KITTI360_ROOT, 'data_3d_bboxes', 'train', f'{KITTI360_SEQ}.xml')
if os.path.isfile(bbox_file):
    print('data_3d_bboxes: already present')
else:
    z = _wget(f'{S3}/ffa164387078f48a20f0188aa31b0384bb19ce60/data_3d_bboxes.zip', KITTI360_ROOT)
    _unzip(z, KITTI360_ROOT, show_contents=True)
    print('data_3d_bboxes: done')

# ── 4. Perspective images — construct URL directly from known S3 pattern ──────
img_dir = os.path.join(KITTI360_ROOT, 'data_2d_raw', KITTI360_SEQ, 'image_00', 'data_rect')
if os.path.isdir(img_dir) and len(os.listdir(img_dir)) > 10:
    print(f'data_2d_raw: already present ({len(os.listdir(img_dir))} frames)')
else:
    # Images zip extracts flat: {seq}/image_00/data_rect/*.png
    # so extract into data_2d_raw/
    dl_dir = os.path.join(KITTI360_ROOT, 'data_2d_raw')
    os.makedirs(dl_dir, exist_ok=True)

    img_url = f'{S3}/data_2d_raw/{KITTI360_SEQ}_image_00.zip'
    print(f'Probing image URL: {img_url}')
    if _probe_url(img_url):
        z = _wget(img_url, dl_dir)
        _unzip(z, dl_dir, show_contents=True)
        n = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
        print(f'Perspective images: done ({n} frames)')
    else:
        # Fallback: download the script zip and print its full contents for inspection
        print('Direct URL not found — checking download script for actual URL...')
        script_zip = _wget(
            f'{S3}/a1d81d9f7fc7195c937f9ad12e2a2c66441ecb4e/download_2d_perspective.zip', '/tmp')
        with zipfile.ZipFile(script_zip) as zf:
            sh_names = [n for n in zf.namelist() if n.endswith('.sh')]
            zf.extractall('/tmp')
        os.remove(script_zip)
        with open(os.path.join('/tmp', sh_names[0])) as f:
            script_content = f.read()
        print('=== download_2d_perspective.sh ===')
        print(script_content)
        print('=== end of script ===')
        raise RuntimeError('Could not resolve image download URL — see script above.')

# ── 5. LiDAR ──────────────────────────────────────────────────────────────────
if DOWNLOAD_LIDAR:
    lidar_dir = os.path.join(KITTI360_ROOT, 'data_3d_raw', KITTI360_SEQ,
                             'velodyne_points', 'data')
    if os.path.isdir(lidar_dir) and len(os.listdir(lidar_dir)) > 10:
        print(f'data_3d_raw: already present ({len(os.listdir(lidar_dir))} scans)')
    else:
        dl_dir = os.path.join(KITTI360_ROOT, 'data_3d_raw')
        os.makedirs(dl_dir, exist_ok=True)

        lidar_url = f'{S3}/data_3d_raw/{KITTI360_SEQ}_velodyne.zip'
        print(f'Probing LiDAR URL: {lidar_url}')
        if _probe_url(lidar_url):
            z = _wget(lidar_url, dl_dir)
            _unzip(z, dl_dir, show_contents=True)
            n = len(os.listdir(lidar_dir)) if os.path.isdir(lidar_dir) else 0
            print(f'LiDAR: done ({n} scans)')
        else:
            print('Direct URL not found — checking download script for actual URL...')
            script_zip = _wget(
                f'{S3}/a1d81d9f7fc7195c937f9ad12e2a2c66441ecb4e/download_3d_velodyne.zip', '/tmp')
            with zipfile.ZipFile(script_zip) as zf:
                sh_names = [n for n in zf.namelist() if n.endswith('.sh')]
                zf.extractall('/tmp')
            os.remove(script_zip)
            with open(os.path.join('/tmp', sh_names[0])) as f:
                script_content = f.read()
            print('=== download_3d_velodyne.sh ===')
            print(script_content)
            print('=== end of script ===')
            raise RuntimeError('Could not resolve LiDAR download URL — see script above.')

# ── 6. Validate ───────────────────────────────────────────────────────────────
checks = {
    'calibration/perspective.txt':              os.path.join(KITTI360_ROOT, 'calibration', 'perspective.txt'),
    'calibration/calib_cam_to_pose.txt':        os.path.join(KITTI360_ROOT, 'calibration', 'calib_cam_to_pose.txt'),
    f'data_poses/{KITTI360_SEQ}/poses.txt':     poses_file,
    f'data_3d_bboxes/train/{KITTI360_SEQ}.xml': bbox_file,
    f'data_2d_raw/{KITTI360_SEQ}/image_00':     img_dir,
}
print('\n=== Validation ===')
all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    n = len(os.listdir(path)) if (exists and os.path.isdir(path)) else ''
    suffix = f' ({n} files)' if n != '' else ''
    print(f'  {"OK" if exists else "MISSING":7s}  {label}{suffix}')
    if not exists:
        all_ok = False

if all_ok:
    print(f'\nReady. Set in cell-pipeline-config:')
    print(f'  KITTI360_ROOT = "{KITTI360_ROOT}"')
    print(f'  KITTI360_SEQ  = "{KITTI360_SEQ}"')
else:
    print('\nSome files missing — check errors above.')


In [ ]:
# Cell 7 — Parse calibration → real 3×3 intrinsics K
if DATA_FORMAT == '360':
    if KITTI360_ROOT is None:
        raise ValueError("Set KITTI360_ROOT in cell-pipeline-config before running this cell.")
    from kitti360scripts.devkits.commons.loadCalibration import loadPerspectiveIntrinsic
    _intr = loadPerspectiveIntrinsic(f'{KITTI360_ROOT}/calibration/perspective.txt')
    K = np.asarray(_intr['P_rect_00'], dtype=np.float32)[:3, :3]
else:
    with open(os.path.join(DATA_PATH_ROOT, 'calib_cam_to_cam.txt')) as f:
        calib_lines = f.readlines()
    P_rect = np.array(
        [float(x) for x in calib_lines[25].strip().split(' ')[1:]]
    ).reshape(3, 4)
    K = P_rect[:3, :3].astype(np.float32)

print('K matrix (3x3 intrinsics):')
print(K)
print(f'  fx={K[0,0]:.2f}  fy={K[1,1]:.2f}  cx={K[0,2]:.2f}  cy={K[1,2]:.2f}')

In [ ]:
# Cell 7b — Load calibration and optionally LiDAR
#
# DATA_FORMAT is set in cell-pipeline-config.
# lidar_seq is set to None if LiDAR data was not downloaded (that is fine —
# the pipeline runs without LiDAR; BEV will use monocular depth only).

import os
from fusion_perception.data.kitti_calibration import load_calibration
from fusion_perception.data.lidar_loader import KittiRawLidar, Kitti360Lidar, load_bin

if DATA_FORMAT == 'raw':
    calib = load_calibration(DATA_PATH_ROOT)
    try:
        lidar_seq = KittiRawLidar.from_dir(DATA_PATH_ROOT)
    except FileNotFoundError:
        lidar_seq = None
        print('LiDAR not found — running without LiDAR')
else:
    if KITTI360_ROOT is None:
        raise ValueError("Set KITTI360_ROOT in cell-pipeline-config.")
    calib = load_calibration(KITTI360_ROOT)
    try:
        lidar_seq = Kitti360Lidar.from_dir(KITTI360_ROOT)
    except FileNotFoundError:
        lidar_seq = None
        print('LiDAR not found — running without LiDAR (set DOWNLOAD_LIDAR=True in cell-kitti360-download to add it)')

print(f'Calibration : {type(calib).__name__}')
print(f'LiDAR       : {len(lidar_seq._files)} frames' if lidar_seq else 'LiDAR       : not available')

if lidar_seq:
    p0 = lidar_seq.get_path(0)
    if p0:
        pts_test = load_bin(p0)
        pts_cam  = calib.velo_to_cam(pts_test)
        print(f'Frame 0     : {len(pts_test)} raw pts -> {len(pts_cam)} front-facing pts in camera frame')


In [ ]:
# Cell 7c -- Load KITTI-360 GT 3D labels (optional -- skip for KITTI raw)

import importlib, pkgutil, re, sys, os

# Patch entire kitti360scripts package for NumPy >= 1.24 (np.int removed)
import kitti360scripts as _k360
_pkg_dir = os.path.dirname(_k360.__file__)
_patched = []
for _finder, _name, _ispkg in pkgutil.walk_packages([_pkg_dir], prefix='kitti360scripts.'):
    try:
        _spec = importlib.util.find_spec(_name)
        if _spec and _spec.origin and _spec.origin.endswith('.py'):
            with open(_spec.origin) as _f:
                _src = _f.read()
            _new = re.sub(r'\bnp\.int\b', 'int', _src)
            if _new != _src:
                with open(_spec.origin, 'w') as _f:
                    _f.write(_new)
                _patched.append(os.path.basename(_spec.origin))
    except Exception:
        pass
for _mod in list(sys.modules):
    if 'kitti360' in _mod:
        del sys.modules[_mod]
if _patched:
    print(f'Patched kitti360scripts files: {_patched}')

from fusion_perception.data.kitti360_labels import Kitti360GTLabels

gt_labels = None
if KITTI360_ROOT is not None:
    gt_labels = Kitti360GTLabels(KITTI360_ROOT, KITTI360_SEQ)
    print(f'GT labels loaded: {gt_labels.annotation.num_bbox} 3D boxes')

    sample = gt_labels.get_boxes_for_frame(0)
    print(f'Frame 0: {len(sample)} GT boxes in camera view')
    for b in sample[:3]:
        cx, cy, cz = b['box3d'][:3]
        cls = b['class']
        iid = b['instance_id']
        print(f'  {cls:15s} id={iid:4d}  cx={cx:.1f} cy={cy:.1f} cz={cz:.1f}m')
else:
    print('KITTI360_ROOT not set -- GT label loading skipped')


In [ ]:
# Cell 8 — Collect image paths and compute FPS
import pandas as pd
from glob import glob

if DATA_FORMAT == 'raw':
    left_imgs = sorted(glob(os.path.join(DATA_PATH_ROOT, DRIVE_SEQ, 'image_02/data/*.png')))
    print(f'Total frames: {len(left_imgs)}')
    ts = pd.read_csv(
        os.path.join(DATA_PATH_ROOT, DRIVE_SEQ, 'image_02/timestamps.txt'),
        header=None
    ).squeeze().astype(object).apply(lambda x: x.split(' ')[1])
    def hms(t):
        h, m, s = t.split(':')
        return int(h)*3600 + int(m)*60 + float(s)
    secs      = ts.apply(hms)
    KITTI_FPS = float(1.0 / np.median(np.diff(secs.values)))
else:
    if KITTI360_ROOT is None:
        raise ValueError("Set KITTI360_ROOT = '/content/drive/MyDrive/Fusion_sensor/KITTI-360' in cell-pipeline-config.")
    _img_dir  = f'{KITTI360_ROOT}/data_2d_raw/{KITTI360_SEQ}/image_00/data_rect'
    if not os.path.isdir(_img_dir):
        raise FileNotFoundError(f"Image directory not found: {_img_dir}Run cell-kitti360-download first.")
    left_imgs = sorted(glob(f'{_img_dir}/*.png'))
    KITTI_FPS = 10.0
    print(f'Total frames: {len(left_imgs)}')

if not left_imgs:
    raise FileNotFoundError("No frames found — check KITTI360_ROOT and KITTI360_SEQ, and confirm cell-kitti360-download completed.")

print(f'FPS: {KITTI_FPS:.2f}')
print(f'First frame: {left_imgs[0]}')


In [ ]:
# Cell 9 — Encode frames to MP4 (optional — set ENCODE_VIDEO=True to run)
import cv2

if not ENCODE_VIDEO:
    print('Video encoding skipped (ENCODE_VIDEO=False). Inference reads frames directly from Drive.')
else:
    paths  = left_imgs if MAX_FRAMES is None else left_imgs[:MAX_FRAMES]
    sample = cv2.imread(paths[0])
    H, W   = sample.shape[:2]
    writer = cv2.VideoWriter(VIDEO_PATH, cv2.VideoWriter_fourcc(*'mp4v'), KITTI_FPS, (W, H))
    for p in paths:
        frame = cv2.imread(p)
        if frame is not None:
            writer.write(frame)
    writer.release()
    print(f'Encoded {len(paths)} frames -> {VIDEO_PATH}  ({W}x{H} @ {KITTI_FPS:.1f} fps)')


In [ ]:
# Cell 10 — Smoke test (CPU only, verifies imports)
from fusion_perception.utils.dataclasses import Track, OccupancyGrid, SceneMemory
from fusion_perception.utils.logging_setup import setup_logging, get_logger
from fusion_perception.occupancy.bev_grid import OccupancyBEVGenerator
from fusion_perception.reasoning.prompt_builder import build_scene_prompt

setup_logging(level='INFO', use_rich=False)
logger = get_logger('smoke')

bev  = OccupancyBEVGenerator(resolution=0.5, x_range=[-20.,20.], z_range=[0.,50.], decay_factor=0.95)
grid = bev.update([], frame_idx=0)
assert len(grid.grid) == 100

track = Track(
    track_id=1, class_name='car', first_seen=0, last_seen=5,
    centroid_history=[[320.,240.]], position_3d_history=[[0.,0.,15.]],
    cow_query_point=[320.,240.], is_active=True, occlusion_count=0,
)
occ = OccupancyGrid(frame_idx=0, resolution=0.5, x_range=[-20.,20.], z_range=[0.,50.],
                    grid=[[0.]*80 for _ in range(100)], decay_factor=0.95)
mem = SceneMemory(frame_idx=0, active_tracks=[track], occupancy_grid=occ,
                  event_flags=[], frame_count=1, elapsed_seconds=0.0)
prompt = build_scene_prompt(mem)
assert 'car' in prompt and K.shape == (3, 3)
print('Smoke test passed')